# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [49]:
%matplotlib ipympl

In [ ]:
import pandas as pd

from analysis.build_data import load_trace_df


In [65]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence")

## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the label the model predicts the market resolves to, `confidence` is the model's stated probability that the label is right, and a forecast is accurate when the label matches the market's `resolution` (ignoring case).

In [68]:
# scoreable forecasts: a predicted resolution with a confidence, on a market that has resolved
calibrated = forecasts.dropna(subset=["correct", "confidence"])

In [69]:
B = 10

In [70]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [71]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

In [72]:
def expected_calibration_error(forecasts: pd.DataFrame) -> float:
    """sum_b |B_b|/N * |acc(B_b) - conf(B_b)| over the confidence bins."""
    bins = forecasts.groupby("confidence_bin", observed=True)
    acc = bins["correct"].mean()
    conf = bins["confidence"].mean()
    weight = bins.size() / len(forecasts)
    return (weight * (acc - conf).abs()).sum()

In [73]:
ece = expected_calibration_error(calibrated)

In [74]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

In [75]:
accuracy_by_model = calibrated.groupby("model")["correct"].mean()

In [76]:
# per model: how often its call was right, and how far its confidence is from that
model_table = pd.DataFrame({"accuracy": accuracy_by_model, "ece": ece_by_model})

In [77]:
model_table

,accuracy,ece
model,,
claude-opus-5-5,0.3,0.509


In [78]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)